# K562_gwps subset validation

Goal: determine whether PertAdapt/GEARS can realistically handle K562_gwps's scale
(1,989,578 cells, 9,866 perturbed genes) before committing to a full run.

Context: K562_essential (what we trained on before) has **zero** gene overlap with the
real 2026 VCC evaluation panel (300 genes). K562_gwps has **277/300 (92.3%)** overlap -
it's the only usable training dataset among what's available, but it's ~6.4x more cells
and ~4.8x more perturbed genes than K562_essential, which is the likely cause of past
issues running the full thing.

This notebook builds a ~500-gene subset (277 real overlap genes + 223 additional genes
for GO-graph context) and runs the full pipeline (DE-ranking, GO-graph, one training
step) on it, instrumented for time/memory, to get real scaling numbers before deciding
whether/how to run the full 9,866-gene dataset.

### 1. Setup

In [ ]:
import os
import subprocess
import sys

gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("WARNING: nvidia-smi failed - no GPU visible on this machine.")
    print(gpu_check.stderr)
else:
    print(gpu_check.stdout)

REPO_ROOT = subprocess.run(
    ['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True, check=True
).stdout.strip()
PERTADAPT_ROOT = f"{REPO_ROOT}/PertAdapt"
DATA_DIR = f"{REPO_ROOT}/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("PERTADAPT_ROOT:", PERTADAPT_ROOT)
print("DATA_DIR:", DATA_DIR)

### 2. Build the K562_gwps subset

Selects 277 genes confirmed to overlap the real 2026 300-gene panel, plus 223 additional
genes (for broader GO-graph context), applies the same scFoundation-vocabulary remapping
pipeline already validated on K562_essential, and saves a subset h5ad.

In [ ]:
import time
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse
import scanpy as sc

t_start = time.time()

gwps_raw_path = f"{DATA_DIR}/K562_gwps_raw_singlecell_01.h5ad"
if not os.path.exists(gwps_raw_path):
    print(f"Downloading K562_gwps raw data -> {gwps_raw_path}")
    os.system(f"gcloud storage cp gs://2026-vcc-llsa/train/K562_gwps_raw_singlecell_01.h5ad {gwps_raw_path}")

real_panel_path = f"{DATA_DIR}/pert_counts_2026_final.csv"
assert os.path.exists(real_panel_path), f"Missing {real_panel_path} - copy the real 2026 pert_counts.csv here first"
real_panel = pd.read_csv(real_panel_path)
real_genes = set(real_panel['target_gene'])
print(f"Real 2026 panel: {len(real_genes)} genes")

In [ ]:
print("Loading K562_gwps (backed, metadata only)...")
adata_meta = ad.read_h5ad(gwps_raw_path, backed='r')
gwps_genes = set(adata_meta.obs['gene'].unique()) - {'non-targeting'}

overlap_genes = sorted(real_genes & gwps_genes)
print(f"Overlap genes available in gwps: {len(overlap_genes)}")

non_overlap_gwps_genes = sorted(gwps_genes - real_genes)
rng = np.random.default_rng(seed=1)
extra_genes = rng.choice(non_overlap_gwps_genes, size=min(223, len(non_overlap_gwps_genes)), replace=False).tolist()

subset_genes = set(overlap_genes) | set(extra_genes)
print(f"Subset target gene count: {len(subset_genes)} ({len(overlap_genes)} real-overlap + {len(extra_genes)} additional)")

keep_mask = adata_meta.obs['gene'].isin(subset_genes | {'non-targeting'})
print(f"Cells selected: {keep_mask.sum()} / {adata_meta.shape[0]}")
adata_meta.file.close()

In [ ]:
print("Loading full subset into memory...")
adata = ad.read_h5ad(gwps_raw_path)
adata = adata[adata.obs['gene'].isin(subset_genes | {'non-targeting'})].copy()
print(f"Subset shape after cell selection: {adata.shape}")

adata.obs['condition'] = adata.obs['gene'].apply(
    lambda g: 'ctrl' if g == 'non-targeting' else f'{g}+ctrl'
)
adata.obs['cell_type'] = 'K562'
assert 'gene_id' in adata.var.columns or adata.var.index.name is not None

# gwps var doesn't have a 'gene_name' column like K562_essential did - its var index
# itself holds gene symbols directly (confirmed during the overlap check earlier).
if 'gene_name' not in adata.var.columns:
    adata.var['gene_name'] = adata.var.index

print(adata.obs['condition'].value_counts().head(10))

In [ ]:
# Same scFoundation-vocabulary remapping validated on K562_essential - unchanged logic.
gene_index_path = f"{PERTADAPT_ROOT}/preprocess/OS_scRNA_gene_index.19264.tsv"
canonical_genes = pd.read_csv(gene_index_path, sep='\t')['gene_name'].tolist()
canonical_gene_to_idx = {g: i for i, g in enumerate(canonical_genes)}

overlap_var_genes, overlap_col_idx = [], []
for j, g in enumerate(adata.var['gene_name'].tolist()):
    if g in canonical_gene_to_idx:
        overlap_var_genes.append(g)
        overlap_col_idx.append(j)
print(f"Genes overlapping scFoundation's vocabulary: {len(overlap_var_genes)} / {adata.shape[1]}")

adata = adata[:, overlap_col_idx].copy()
X = scipy.sparse.csr_matrix(adata.X)
old_col_to_new_col = np.array([canonical_gene_to_idx[g] for g in overlap_var_genes], dtype=np.int32)
remapped_indices = old_col_to_new_col[X.indices]
X_remapped = scipy.sparse.csr_matrix(
    (X.data, remapped_indices, X.indptr), shape=(X.shape[0], len(canonical_genes))
)
X_remapped.sum_duplicates()
X_remapped.sort_indices()

adata = ad.AnnData(X=X_remapped, obs=adata.obs)
adata.var = pd.DataFrame(index=canonical_genes, data={'gene_name': canonical_genes})

sc.pp.filter_cells(adata, min_genes=200)
print(f"Shape after min_genes filter: {adata.shape}")

total_count = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs['total_count'] = np.round(total_count).astype(np.float32)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

import anndata as ad_module
ad_module.settings.allow_write_nullable_strings = True

data_name = 'k562_gwps_subset'
data_h5ad_path = f"{PERTADAPT_ROOT}/data/{data_name}.h5ad"
os.makedirs(os.path.dirname(data_h5ad_path), exist_ok=True)
adata.write_h5ad(data_h5ad_path)

t_preprocess = time.time() - t_start
print(f"Saved subset to {data_h5ad_path}")
print(f"Final subset: {adata.shape[0]} cells, {adata.obs['condition'].nunique()} conditions")
print(f"Preprocessing took {t_preprocess:.1f}s")

### 3. Run the GEARS pipeline on the subset, instrumented

Same structure as the original preflight script: filter gene2go.pkl, time GO-graph
construction (model_initialize), run one real training step. This is a standalone
script (bypasses torchrun) so print() output isn't swallowed.

In [ ]:
preflight_script = r'''
import os
import sys
import time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, '{PERTADAPT_ROOT}/scFoundation/PertAdapter')

import torch
import torch.distributed as dist
from os.path import join as pjoin

os.environ['ANNDATA_ALLOW_WRITE_NULLABLE_STRINGS'] = '1'
os.environ.setdefault('MASTER_ADDR', '127.0.0.1')
os.environ.setdefault('MASTER_PORT', '29503')
dist.init_process_group(backend='gloo', rank=0, world_size=1)
print("PREFLIGHT: process group initialized", flush=True)

device = 'cuda:0'
torch.cuda.set_device(device)

from gears import PertData, GEARSAdapt_ddp

data_dir = '{PERTADAPT_ROOT}/data/'
data_name = 'k562_gwps_subset'

import pickle
gene2go_pkl_path = os.path.join(data_dir, 'gene2go.pkl')
gene2go_full_path = os.path.join(data_dir, 'gene2go_full.pkl')
import anndata as ad
adata_obs = ad.read_h5ad(pjoin(data_dir, data_name + '.h5ad'), backed='r')
target_genes = {{c.split('+')[0] for c in adata_obs.obs['condition'].unique() if c != 'ctrl'}}
adata_obs.file.close()
with open(gene2go_full_path, 'rb') as f:
    gene2go_full = pickle.load(f)
gene2go_filtered = {{g: v for g, v in gene2go_full.items() if g in target_genes}}
print(f"PREFLIGHT: gene2go filtered to {{len(gene2go_filtered)}} genes (target set: {{len(target_genes)}})", flush=True)
with open(gene2go_pkl_path, 'wb') as f:
    pickle.dump(gene2go_filtered, f)

import resource
def peak_rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6

t0 = time.time()
pert_data = PertData(data_dir)
try:
    pert_data.load(data_path=pjoin(data_dir, data_name))
except Exception:
    adata = ad.read_h5ad(pjoin(data_dir, data_name + '.h5ad'))
    adata.uns['log1p'] = {{}}
    adata.uns['log1p']['base'] = None
    pert_data.new_data_process(dataset_name=data_name, adata=adata)
    pert_data.load(data_path=pjoin(data_dir, data_name))
t_data = time.time() - t0
print(f"PREFLIGHT: data pipeline (DE-ranking + graph building) took {{t_data:.1f}}s, peak RSS so far: {{peak_rss_gb():.2f}} GB", flush=True)

pert_data.prepare_split(split='simulation', seed=1, train_gene_set_size=0.75)
BATCH_SIZE = 2
pert_data.get_dataloader(batch_size=BATCH_SIZE, test_batch_size=BATCH_SIZE)
print("PREFLIGHT: dataloaders ready", flush=True)

gears_model = GEARSAdapt_ddp(
    pert_data, local_rank=0, is_master=True, world_size=1,
    device=device, weight_bias_track=False, train_bs=BATCH_SIZE, test_bs=BATCH_SIZE,
    proj_name='preflight_gwps', exp_name='preflight_gwps', loss='loss_adapt',
)

t1 = time.time()
gears_model.model_initialize(
    hidden_size=512, model_type='maeautobin', bin_set='autobin_resolution_append',
    load_path='{PERTADAPT_ROOT}/scFoundation/model/models/models.ckpt',
    finetune_method='frozen', accumulation_steps=2, mode='v1', highres=0,
    model_class='GEARS_Model_Pert_Adapter_New', de_loss_weight=0.5,
)
t_init = time.time() - t1
print(f"PREFLIGHT: model_initialize (incl. GO-graph) took {{t_init:.1f}}s, peak RSS: {{peak_rss_gb():.2f}} GB", flush=True)

model_devices = {{p.device for p in gears_model.model.parameters()}}
print(f"PREFLIGHT: model parameter devices: {{model_devices}}", flush=True)
assert model_devices == {{torch.device('cuda:0')}}

train_loader = pert_data.dataloader['train_loader']
batch = next(iter(train_loader))
batch.to(device)

from gears.utils import loss_adapt
opt = torch.optim.Adam(gears_model.model.parameters(), lr=0.01)
gears_model.model.train()
if hasattr(gears_model.model, 'singlecell_model'):
    gears_model.model.singlecell_model.eval()

torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
t2 = time.time()

pred = gears_model.model(batch)
y = batch.y
loss = loss_adapt(pred, y, batch.pert, gears_model.adata, gears_model.geneid2idx,
                   gears_model.pert2pert_full_id, gears_model.dict_filter,
                   de_loss_weight=gears_model.config['de_loss_weight'])
loss.backward()
opt.step()
opt.zero_grad()

torch.cuda.synchronize()
t_step = time.time() - t2
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"PREFLIGHT: one training step (fwd+bwd+opt) took {{t_step:.3f}}s", flush=True)
print(f"PREFLIGHT: loss value: {{loss.item():.4f}}", flush=True)
print(f"PREFLIGHT: peak GPU memory during step: {{peak_mem_gb:.2f}} GB", flush=True)
print(f"PREFLIGHT: peak host RSS overall: {{peak_rss_gb():.2f}} GB", flush=True)
print(f"PREFLIGHT: n_perturbed_genes_in_subset: {{len(target_genes)}}", flush=True)
print(f"PREFLIGHT: n_cells_in_subset: {{adata_obs.shape[0] if hasattr(adata_obs, 'shape') else 'n/a'}}", flush=True)
print("PREFLIGHT: ALL CHECKS PASSED - GWPS SUBSET PIPELINE VERIFIED END TO END", flush=True)
'''.format(PERTADAPT_ROOT=PERTADAPT_ROOT)

script_path = '/tmp/preflight_gwps_subset.py'
with open(script_path, 'w') as f:
    f.write(preflight_script)

print(f"Wrote preflight script to {script_path}")
print(f"Run it with: {sys.executable} {script_path} 2>&1 | tee /tmp/preflight_gwps_subset.log")

In [ ]:
import subprocess
log_path = '/tmp/preflight_gwps_subset.log'
with open(log_path, 'w') as logf:
    result = subprocess.run(
        [sys.executable, script_path],
        stdout=logf, stderr=subprocess.STDOUT,
    )
print(f"Exit code: {result.returncode}")
print(f"--- {log_path} ---")
with open(log_path) as f:
    print(f.read())

### 4. Extrapolate to full-scale K562_gwps

Once the cell above completes, use the printed timings to extrapolate:
- GO-graph/DE-ranking time scaling to the full 9,866 genes
- Peak host RAM scaling to the full 1,989,578 cells
- Per-step GPU memory/time scaling for the full training run

This tells us whether the full run is actually feasible, and at what batch size,
before committing real GPU-hours to it.